In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt  
import seaborn as sns

In [3]:
df=pd.read_csv('./data/premium.csv')
df.head()

,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520


In [4]:
df=df.drop_duplicates()
df.info()

<class 'pandas.DataFrame'>
Index: 1337 entries, 0 to 1337
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   age       1337 non-null   int64  
 1   sex       1337 non-null   str    
 2   bmi       1332 non-null   float64
 3   children  1337 non-null   int64  
 4   smoker    1337 non-null   str    
 5   region    1337 non-null   str    
 6   charges   1337 non-null   float64
dtypes: float64(2), int64(2), str(3)
memory usage: 83.6 KB


In [5]:
df['bmi']=df['bmi'].fillna(df['bmi'].mean())
df.isnull().sum()


age         0
sex         0
bmi         0
children    0
smoker      0
region      0
charges     0
dtype: int64

In [6]:
from sklearn.preprocessing import LabelEncoder

In [9]:
# 문자열 데이터의 수치화 > LabelEncoder
col_list =['sex','smoker','region']
for col in col_list:
  enc = LabelEncoder()
  df[col] = enc.fit_transform(df[col])
df.head()

,age,sex,bmi,children,smoker,region,charges
0,19,0,27.900,0,1,3,16884.92400
1,18,1,33.770,1,0,2,1725.55230
2,28,1,33.000,3,0,2,4449.46200
3,33,1,22.705,0,0,1,21984.47061
4,32,1,28.880,0,0,1,3866.85520


In [12]:
# 스케일링 하기 
df['charges_log'] = np.log1p(df['charges'])
df['charges_log']

0        9.734236
1        7.453882
2        8.400763
3        9.998137
4        8.260455
          ...    
1333     9.268755
1334     7.699381
1335     7.396847
1336     7.605365
1337    10.279948
Name: charges_log, Length: 1337, dtype: float64

In [13]:
X=df.iloc[:,:-1]
y=df.iloc[:,:-1]

In [16]:
from sklearn.model_selection import train_test_split 
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)

In [17]:
from sklearn.preprocessing import MinMaxScaler
scaler= MinMaxScaler()
X_train_sclaed=scaler.fit_transform(X_train)
X_test_sclaed =scaler.fit_transform(X_test)

In [20]:
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score

# 1. 데이터 준비 (전달해주신 이미지 컬럼 기
X = df[['age', 'sex', 'bmi', 'children', 'smoker', 'region']]
y = df['charges']

# 타겟 변수 로그 변환 (수치적 안정성 확보)
y_log = np.log1p(y)

# 2. 전처리 전략 수립 (ColumnTransformer)
# BMI와 Age는 표준화(StandardScaler) 적용, 나머지는 통과(passthrough)
numeric_features = ['age', 'bmi', 'children']
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features)
    ], 
    remainder='passthrough'
)

# 3. 모델 리스트
models = [
    ('Linear', LinearRegression()),
    ('Ridge', Ridge(alpha=1.0)),
    ('Lasso', Lasso(alpha=0.01)),
    ('ElasticNet', ElasticNet(alpha=0.01, l1_ratio=0.5))
]

# 4. For 문을 이용한 통합 학습
X_train, X_test, y_train, y_test = train_test_split(X, y_log, test_size=0.2, random_state=42)

print(f"{'Model':<12} | {'RMSE':<10} | {'R2 Score':<10}")
print("-" * 38)

for name, reg in models:
    # 파이프라인 구성: [특정컬럼 스케일링 -> 다항 특성 추가 -> 회귀 모델]
    pipe = Pipeline([
        ('preprocessor', preprocessor),
        ('poly', PolynomialFeatures(degree=2, include_bias=False)),
        ('regressor', reg)
    ])
    
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)
    
    print(f"{name:<12} | {rmse:<10.4f} | {r2:<10.4f}")

Model        | RMSE       | R2 Score  
--------------------------------------
Linear       | 0.3240     | 0.8869    
Ridge        | 0.3241     | 0.8868    
Lasso        | 0.3371     | 0.8775    
ElasticNet   | 0.3310     | 0.8819    


In [ ]:
# 선형회귀 모델 
from sklearn.linear_model import LinearRegression
lr_model = LinearRegression()
lr_model.fit(X_train,y_train) 

In [21]:
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.linear_model import Ridge, ElasticNet
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score

# 1. 데이터 준비 및 로그 변환
# X = df[['age', 'sex', 'bmi', 'children', 'smoker', 'region']]
y_log = np.log1p(df['charges'])
X_train, X_test, y_train, y_test = train_test_split(X, y_log, test_size=0.2, random_state=42)

# 2. 전처리 설정 (BMI, Age 등 수치형만 스케일링)
numeric_features = ['age', 'bmi', 'children']
preprocessor = ColumnTransformer([
    ('num', StandardScaler(), numeric_features)
], remainder='passthrough')

# 3. 실험할 차수와 모델 정의
degrees = [1, 2, 3]  # 1차(선형), 2차, 3차 비교
models = [
    ('Ridge', Ridge(alpha=1.0)),
    ('ElasticNet', ElasticNet(alpha=0.01))
]

print(f"{'Model':<12} | {'Deg':<3} | {'RMSE':<10} | {'R2 Score':<10}")
print("-" * 45)

# 4. 중첩 for문: 모델별로 차수를 바꿔가며 실행
for model_name, model in models:
    for deg in degrees:
        # 파이프라인 구성
        pipe = Pipeline([
            ('preprocessor', preprocessor),
            ('poly', PolynomialFeatures(degree=deg, include_bias=False)),
            ('regressor', model)
        ])
        
        pipe.fit(X_train, y_train)
        y_pred = pipe.predict(X_test)
        
        rmse = np.sqrt(mean_squared_error(y_test, y_pred))
        r2 = r2_score(y_test, y_pred)
        
        print(f"{model_name:<12} | {deg:<3} | {rmse:<10.4f} | {r2:<10.4f}")

Model        | Deg | RMSE       | R2 Score  
---------------------------------------------
Ridge        | 1   | 0.4002     | 0.8274    
Ridge        | 2   | 0.3241     | 0.8868    
Ridge        | 3   | 0.3385     | 0.8765    
ElasticNet   | 1   | 0.4057     | 0.8227    
ElasticNet   | 2   | 0.3310     | 0.8819    
ElasticNet   | 3   | 0.3366     | 0.8779    
